# FailurePostprocessor + VLMService.select_checkpoint_index Test

This notebook tests the VLM-based checkpoint selection functionality.

In [ ]:
import json
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.failure_postprocessor import FailurePostprocessor

cwd = Path.cwd().resolve()

# View key mapping
VIEW_KEY_MAP = {
    "left": "observation.images.left",
    "middle": "observation.images.middle",
    "right": "observation.images.right",
}


# Helper functions
def _to_chw_float_tensor(img_path: Path, target_hw: tuple[int, int] | None = None) -> torch.Tensor:
    """Load image from path and convert to CHW float tensor."""
    img = Image.open(img_path).convert("RGB")
    if target_hw is not None:
        target_h, target_w = target_hw
        img = img.resize((target_w, target_h), Image.Resampling.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1).contiguous()
    return tensor


def _infer_target_hw(batch_item: dict) -> tuple[int, int] | None:
    """Infer target (height, width) from batch item images."""
    for key in VIEW_KEY_MAP.values():
        if key in batch_item and isinstance(batch_item[key], torch.Tensor):
            t = batch_item[key]
            if t.dim() == 4 and t.size(0) >= 1:
                t = t[0]
            if t.dim() == 3:
                return int(t.shape[-2]), int(t.shape[-1])
    return None


def _collect_three_view_paths(step_dir: Path) -> dict[str, Path]:
    """Collect left/middle/right image paths from a step directory."""
    paths = {cam: step_dir / f"{cam}.png" for cam in ["left", "middle", "right"]}
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(f"time step={step_dir.name} missing view images: {missing}")
    return paths


print("✓ Imports and helper functions loaded")

In [ ]:
# ===== Dataset Loading =====
repo_id = "eval/eval_pick_up_markers_failure_test2"
sample_index = 0

dataset = LeRobotDataset(repo_id, episodes=[0])
item = dataset[sample_index]

# Infer target image size from the dataset
target_hw = _infer_target_hw(item)

print(f"✓ Dataset loaded: {repo_id}")
print(f"  Sample index: {sample_index}")
print(f"  Target image size (H, W): {target_hw}")

In [ ]:
# ===== Initialize FailurePostprocessor =====
task_dir = cwd / "examples" / "pick_up_markers"
cfg_path = task_dir / "failure_handling.json"

if not cfg_path.exists():
    raise FileNotFoundError(f"Missing config file: {cfg_path}")

with cfg_path.open("r", encoding="utf-8") as f:
    cfg_raw = json.load(f)

demo_video_path = Path(cfg_raw["demo_video_path"]).expanduser().resolve()
if not demo_video_path.exists():
    raise FileNotFoundError(f"demo_video_path does not exist: {demo_video_path}")

post = FailurePostprocessor(
    policy=None,
    output_dir=None,
    failure_handling_json_path=cfg_path,
    enable_logging=False,
)

print("✓ FailurePostprocessor initialized")
print(f"  Config: {cfg_path}")
print(f"  Demo video: {demo_video_path}")

In [ ]:
# ===== Load Example Images and Prepare Data =====
example_id = "2"
example_dirs = task_dir / example_id
failure_root = example_dirs / "failure"
checkpoint_root = example_dirs / "checkpoints"

if not failure_root.exists():
    raise FileNotFoundError(f"Missing failure directory: {failure_root}")
if not checkpoint_root.exists():
    raise FileNotFoundError(f"Missing checkpoints directory: {checkpoint_root}")

# Collect all failure and checkpoint images
image_paths_by_kind_step = {"failure": {}, "checkpoints": {}}
for kind, root in [("failure", failure_root), ("checkpoints", checkpoint_root)]:
    for step_dir in sorted(root.iterdir()):
        if not step_dir.is_dir():
            continue
        try:
            step = int(step_dir.name)
        except ValueError:
            continue
        image_paths_by_kind_step[kind][step] = _collect_three_view_paths(step_dir)

if not image_paths_by_kind_step["failure"]:
    raise ValueError("No failure images found.")
if not image_paths_by_kind_step["checkpoints"]:
    raise ValueError("No checkpoint images found.")

# 1) Replace the three image tensors in the current batch with failure-step images
failure_steps = sorted(image_paths_by_kind_step["failure"].keys())
selected_failure_step = failure_steps[-1]  # Use the last failure step

batch_for_vlm = deepcopy(item)
for view_name, tensor_key in VIEW_KEY_MAP.items():
    img_path = image_paths_by_kind_step["failure"][selected_failure_step][view_name]
    batch_for_vlm[tensor_key] = _to_chw_float_tensor(img_path, target_hw=target_hw)

# 2) Build checkpoint_queue: (checkpoint_step, action, checkpoint_views)
checkpoint_queue = []
for step_idx in sorted(image_paths_by_kind_step["checkpoints"].keys()):
    checkpoint_views = {}
    for view_name, tensor_key in VIEW_KEY_MAP.items():
        img_path = image_paths_by_kind_step["checkpoints"][step_idx][view_name]
        checkpoint_views[tensor_key] = _to_chw_float_tensor(img_path, target_hw=target_hw)
    checkpoint_queue.append((int(step_idx), None, checkpoint_views))

print(f"✓ Example data loaded: {example_id}")
print(f"  Failure steps: {failure_steps}")
print(f"  Selected failure step: {selected_failure_step}")
print(f"  Checkpoint queue size: {len(checkpoint_queue)}")
print(f"  Checkpoint steps: {[entry[0] for entry in checkpoint_queue]}")

In [ ]:
selected_index = post.vlm_service.select_checkpoint_index(
    batch=batch_for_vlm,
    checkpoint_queue=checkpoint_queue,
    episode=0,
    step=selected_failure_step,
)
print(f"selected_index={selected_index}, selected_step={checkpoint_queue[selected_index][0]}")
saved_dir = post.vlm_service.save_debug_history()

In [ ]:
import json
from pathlib import Path

from IPython import display

saved_path = Path(saved_dir)
manifest_path = saved_path / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"manifest.json not found in: {saved_path}")

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

records = manifest.get("records", [])
if not records:
    raise ValueError(f"No records found in manifest: {manifest_path}")

print(f"Loaded {len(records)} record(s) from: {saved_path}\n")

for rec in sorted(records, key=lambda x: int(x.get("index", 0))):
    rec_index = int(rec.get("index", 0))
    rec_dir = saved_path / rec["dir"]
    meta_path = rec_dir / "meta.json"
    if not meta_path.exists():
        print(f"[Skip] missing meta.json: {rec_dir}")
        continue

    with meta_path.open("r", encoding="utf-8") as f:
        meta = json.load(f)

    parts = meta.get("request_parts", [])
    response_path = rec_dir / "response.txt"
    has_response = response_path.exists()

    # Display record header
    print("=" * 80)
    print(
        f"Record #{rec_index} | Episode={meta.get('episode')} | Step={meta.get('step')} | Selected={meta.get('selected_index')}"
    )
    print("=" * 80)

    # Display parts
    is_failure = True
    candidate_idx = 0
    for part in parts:
        part_type = part.get("type")
        part_file = part.get("file")
        part_path = rec_dir / part_file

        if part_type == "image" and part_path.exists():
            if is_failure:
                print("\n🔴 Failure State:")
                is_failure = False
            else:
                print(f"\n✓ Candidate #{candidate_idx}:")
                candidate_idx += 1

            img = Image.open(part_path).convert("RGB")
            display.display(img)
        else:
            # Display text content
            if part_path.exists():
                text = part_path.read_text(encoding="utf-8")
                print("\n📝 Text part:")
                print(text)
            else:
                print(f"\n⚠️ [Missing file] {part_file}")

    # Display response
    if has_response:
        response_text = response_path.read_text(encoding="utf-8")
        print("\n💬 Response:")
        print(response_text)

    print("\n")